# Import Library

In [ ]:
from pathlib import Path
import shutil
import os
from tqdm import tqdm
import pandas as pd

# Input Dataframe Awal

In [19]:
df = pd.read_csv("data_downloads_cleaned.csv")

In [20]:
df['FileType'].value_counts()

FileType
gis             2679
image           1049
document         701
data             673
unknown          547
system           375
web              336
spreadsheet      270
text             244
package          216
project          171
video             68
archive           65
audio             49
cad               33
design            26
config            17
code              16
presentation      15
geoscience        11
notebook           3
security           2
subtitle           2
Name: count, dtype: int64

# Definisikan Path Tujuan

In [ ]:
# destination_dir = Path(r"C:\Users\ASUS\OneDrive\Documents\Download-Folder-Management") # CUSTOMIZE
# HARUS DIGANTI! KARENA HARUS TERPISAH DENGAN DOWNLOAD

# Definisi Kelompok FileType di `df` yang Termasuk dalam MainFolder-SubFolder destination_dir

## Pembuatan Struktur Folder Baru

In [ ]:
filetype_groups = {
    "Dokumen & Teks": ["document", "text", "spreadsheet", "presentation", "notebook", "subtitle"],
    "Web & Kode": ["web", "code", "config", "package", "project"],
    "Media": ["image", "video", "audio", "design", "cad"],
    "GIS & Geoscience": ["gis", "geoscience"],
    "Data & Sistem": ["data", "system", "archive", "security"],
    "Lainnya": ["unknown"]
}

# Validasi Kolom Wajib

In [ ]:
required_cols = {"Path", "FileName", "FileExt", "FileType"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"DataFrame tidak memiliki kolom wajib: {required_cols - set(df.columns)}")

# Fungsi Mapping Grup dan Subfolder

In [ ]:
def assign_group_and_subfolder(filetype):
    if pd.isnull(filetype):
        return pd.Series(["Lainnya", "Unknown"])
    for group, types in filetype_groups.items():
        if filetype in types:
            return pd.Series([group, filetype.capitalize()])
    return pd.Series(["Lainnya", filetype.capitalize()])

# Tambahkan Kolom MainFolder & Subfolder di Dataframe

In [ ]:
df[["MainFolder", "Subfolder"]] = df["FileType"].apply(assign_group_and_subfolder)

# Penentuan Path Tujuan Akhir

In [ ]:
df["DestPath"] = df.apply(
    lambda row: destination_dir / row["MainFolder"] / row["Subfolder"] / row["FileName"],
    axis=1
)

# Pembuatan Folder Tujuan

In [ ]:
for folder in df["DestPath"].map(lambda x: x.parent).unique():
    folder.mkdir(parents=True, exist_ok=True)

# Pemindahan File

In [ ]:
print("🚚 Memindahkan file...")
for _, row in tqdm(df.iterrows(), total=len(df)):
    try:
        source = Path(row["Path"])
        target = Path(row["DestPath"])
        if source.exists():
            shutil.move(str(source), str(target))
    except Exception as e:
        print(f"❌ Gagal memindahkan: {row['Path']} -> {row['DestPath']} | Error: {e}")

# Penentuan Folder yang Akan Dipindahkan

`pembuatan folder baru ini agar dipisahkan dengan root_folder agar tidak terjadi looping`

In [ ]:
# downloads_dir = Path(r"C:\Users\ASUS\Downloads")

# Hapus Folder Kosong di Path File yang Telah Dipindahkan

In [ ]:
print("🧹 Menghapus folder kosong di Downloads...")
for dirpath, dirnames, filenames in os.walk(downloads_dir, topdown=False):
    path = Path(dirpath)
    try:
        if not any(path.iterdir()):
            path.rmdir()
            print(f"🧹 Folder kosong dihapus: {path}")
    except Exception as e:
        print(f"⚠️ Gagal menghapus folder: {path} | Error: {e}")